In [6]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [7]:
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD_2"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [8]:
fecha_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=1", con=connection2)
fecha_ini= fecha_ini.iloc[0, 0]

fecha_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_mod=1", con=connection2)
fecha_fin= fecha_fin.iloc[0, 0]

In [9]:
dias_por_intervalo = 15

# Inicializa la fecha actual
fecha_actual = fecha_ini

while fecha_actual <= fecha_fin:
	inicioTiempo = time.time()
	now_inicio = datetime.now()

	fecha_ini_intervalo = fecha_actual
	fecha_fin_intervalo = fecha_actual + timedelta(days=dias_por_intervalo - 1)

	if fecha_fin_intervalo > fecha_fin:
		fecha_fin_intervalo = fecha_fin

	fecha_ini_str = fecha_ini_intervalo.strftime('%Y-%m-%d')
	fecha_fin_str = (fecha_fin_intervalo + timedelta(days=1)).strftime('%Y-%m-%d')
	fecha_fin_display_str = fecha_fin_intervalo.strftime('%Y-%m-%d')

	print(f"Procesando de {fecha_ini_str} al {fecha_fin_display_str}")

	now1 = datetime.now()
	now2 = datetime.now().strftime('%Y-%m-%d')

	query=f"UPDATE etl_act SET fec_act ='{now1}' WHERE id_mod=1"

	c1= text(query)
	connection2.execute(c1)

	tabla='ctcam10'
	col_tabla = "citambproconfec"
	dat= "cext01_essi"
	col_dat='fec_cit'


	oracledb.init_oracle_client()
	oracledb.version = "8.3.0"
	sys.modules["cx_Oracle"] = oracledb
	un = config("USER4_BDI_POSTGRES")
	pw = config("PASS4_BDI_POSTGRES")
	hostname=config("HOST4_BDI_POSTGRES")
	service_name="WNET"
	port = 1527

	engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
			"host": hostname,
			"port": port,
			"service_name": service_name
		}
	)

	connection0 = engine0.connect()


	query1 = f"""
	SELECT
		c.CITAMBORICENASICOD,
		c.CITAMBCENASICOD,
		c.CITAMBNUM,
		c.CITAMBAREHOSCOD,
		c.CITAMBSERVHOSCOD,
		c.CITAMBACTCOD,
		c.CITAMBACTESPCOD,
		c.CITAMBTIPDOCIDENPERCOD,
		c.CITAMBPERASISDOCIDENNUM,
		c.CITAMBPROCONFEC,
		c.CITAMBPROCONTURHORINI,
		c.CITAMBPROCONTURHORFIN,
		c.TIPOCITACOD,
		c.CONDCITACOD,
		c.MODOTORCITACOD,
		c.CITAMBNUMORD,
		c.CITAMBHORCIT,
		c.CITAMBREP,
		c.ESTCITCOD,
		c.ESTCITOTOCOD,
		c.CITAMBUSUCRECOD,
		c.CITAMBCREFEC,
		c.CITAMBUSUMODCOD,
		c.CITAMBMODFEC,
		c.CITAMBSOLFEC,
		c.CITAMBPACSECNUM,
		c.CITAMBUSUANUCOD,
		c.CITAMBANUFEC,
		c.MOTELICITCOD
	FROM {tabla} c
	WHERE c.{col_tabla} >= TO_DATE('{fecha_ini_str}', 'YYYY-MM-DD') AND c.{col_tabla} < TO_DATE('{fecha_fin_str}', 'YYYY-MM-DD')
	"""

	base1 = pd.read_sql_query(query1, con=connection0)
	connection0.close()
	engine0.dispose()

	base1 = base1.rename(columns={
		'citamboricenasicod': 'cod_ori',
		'citambcenasicod': 'cod_cas',
		'citambnum': 'cit_num',
		'citambarehoscod': 'cod_are',
		'citambservhoscod': 'cod_ser',
		'citambactcod': 'cod_act',
		'citambactespcod': 'cod_sub',
		'citambtipdocidenpercod': 'cod_tdo',
		'citambperasisdocidennum': 'num_doc_med',
		'citambproconfec': 'fec_cit',
		'citambproconturhorini': 'hor_ini',
		'citambproconturhorfin': 'hor_fin',
		'tipocitacod': 'cod_tci',
		'condcitacod': 'cod_cci',
		'modotorcitacod': 'cod_emi',
		'citambnumord': 'ord_cit',
		'citambhorcit': 'hor_cit',
		'citambrep': 'cit_rep',
		'estcitcod': 'cod_eci',
		'estcitotocod': 'cod_eco',
		'citambusucrecod': 'usu_cre',
		'citambcrefec': 'fec_cre',
		'citambusumodcod': 'usu_mod',
		'citambmodfec': 'fec_mod',
		'citambsolfec': 'fec_sol',
		'citambpacsecnum': 'pac_sec',
		'citambusuanucod': 'usu_anu',
		'citambanufec': 'fec_anu',
		'motelicitcod': 'cod_eli'
	})

	base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

	oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
	oricas['ori_cod'] = oricas['ori_cod'].str.strip()
	base1['cod_ori'] = base1['cod_ori'].str.strip()
	oricas=oricas.rename(columns={"ori_cod":"cod_ori"})
	base1=pd.merge(base1,oricas,how='left',on='cod_ori')

	base1=base1.drop('cod_ori',axis=1)

	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	base1['cod_cas'] = base1['cod_cas'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas')

	base1=base1.drop('cod_cas',axis=1)

	red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
	red['cod_red'] = red['cod_red'].str.strip()
	base1['cod_red'] = base1['cod_red'].str.strip()
	base1=pd.merge(base1,red,how='left',on='cod_red')

	base1=base1.drop('cod_red',axis=1)

	are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
	are['cod_are'] = are['cod_are'].str.strip()
	base1['cod_are'] = base1['cod_are'].str.strip()
	base1=pd.merge(base1,are,how='left',on='cod_are')

	base1=base1.drop('cod_are',axis=1)

	serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
	serv['cod_ser'] = serv['cod_ser'].str.strip()
	base1['cod_ser'] = base1['cod_ser'].str.strip()
	base1=pd.merge(base1,serv,how='left',on='cod_ser')

	base1=base1.drop('cod_ser',axis=1)

	activi = pd.read_sql_query(f"SELECT id_activi,cod_act FROM dim_activi", con=connection2)
	activi['cod_act'] = activi['cod_act'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	activi=activi.rename(columns={"id_activi":"id_acti"})
	base1=pd.merge(base1,activi,how='left',on='cod_act')

	subacti = pd.read_sql_query(f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti", con=connection2)
	subacti['cod_act'] = subacti['cod_act'].str.strip()
	subacti['cod_sub'] = subacti['cod_sub'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	base1['cod_sub'] = base1['cod_sub'].str.strip()
	subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
	subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
	base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
	base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
	subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
	base1 = pd.merge(base1,subacti,how='left',on='KEY')

	base1=base1.drop('KEY', axis=1)
	base1=base1.drop('cod_act',axis=1)
	base1=base1.drop('cod_sub',axis=1)

	tipdoc = pd.read_sql_query(f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc", con=connection2)
	tipdoc['cod_tdo'] = tipdoc['cod_tdo'].str.strip()
	base1['cod_tdo'] = base1['cod_tdo'].str.strip()
	tipdoc=tipdoc.rename(columns={"id_tipdoc":"id_tdi_med"})
	base1=pd.merge(base1,tipdoc,how='left',on='cod_tdo')

	base1=base1.drop('cod_tdo', axis=1)

	tipcit = pd.read_sql_query(f"SELECT id_tipcit,cod_tci FROM dim_tipcit", con=connection2)
	tipcit['cod_tci'] = tipcit['cod_tci'].str.strip()
	base1['cod_tci'] = base1['cod_tci'].str.strip()
	base1=pd.merge(base1,tipcit,how='left',on='cod_tci')

	base1=base1.drop('cod_tci', axis=1)

	tipcita = pd.read_sql_query(f"SELECT id_condcit,cod_cci FROM dim_condcit", con=connection2)
	tipcita['cod_cci'] = tipcita['cod_cci'].str.strip()
	base1['cod_cci'] = base1['cod_cci'].str.strip()
	base1=pd.merge(base1,tipcita,how='left',on='cod_cci')

	base1=base1.drop('cod_cci', axis=1)

	tipemi = pd.read_sql_query(f"SELECT id_tipemi,cod_emi FROM dim_tipemi", con=connection2)
	tipemi['cod_emi'] = tipemi['cod_emi'].str.strip()
	base1['cod_emi'] = base1['cod_emi'].str.strip()
	base1=pd.merge(base1,tipemi,how='left',on='cod_emi')

	base1=base1.drop('cod_emi', axis=1)

	eci = pd.read_sql_query(f"SELECT id_estcit,cod_eci FROM dim_estcit", con=connection2)
	eci['cod_eci'] = eci['cod_eci'].str.strip()
	base1['cod_eci'] = base1['cod_eci'].str.strip()
	base1=pd.merge(base1,eci,how='left',on='cod_eci')

	base1=base1.drop('cod_eci', axis=1)

	eco = pd.read_sql_query(f"SELECT id_esteco,cod_eco FROM dim_cexcitoto", con=connection2)
	eco['cod_eco'] = eco['cod_eco'].str.strip()
	base1['cod_eco'] = base1['cod_eco'].str.strip()
	base1=pd.merge(base1,eco,how='left',on='cod_eco')

	base1=base1.drop('cod_eco', axis=1)

	moteli = pd.read_sql_query(f"SELECT id_moteli,cod_eli FROM dim_cexmotelicit", con=connection2)
	moteli['cod_eli'] = moteli['cod_eli'].str.strip()
	base1['cod_eli'] = base1['cod_eli'].str.strip()
	base1=pd.merge(base1,moteli,how='left',on='cod_eli')

	base1=base1.drop('cod_eli', axis=1)

	df1_columns = set(base1.columns)
	df2_columns = set(base2.columns) 
	different_columns = df2_columns - df1_columns
	different_columns

	borrando = f"DELETE FROM {dat} WHERE {col_dat} >='{fecha_ini_str}' and {col_dat} <'{fecha_fin_str}'"
	borrado = connection2.execute(borrando)

	comunes = set(base1.columns).intersection(set(base2.columns)) 
	base1 = base1[list(comunes)]
	base1.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=500000)

	fecha_actual = fecha_fin_intervalo + timedelta(days=1)

	finproceso = time.time()
	tiempoproceso = finproceso - inicioTiempo
	tiempoproceso = round(tiempoproceso, 3)
	print("Proceso completado satisfactoriamente en " + str(tiempoproceso) + " segundos")

query2=f"UPDATE etl_act SET fec_ini ='{now2}' WHERE id_mod=1"
c2= text(query2)
connection2.execute(c2)


Procesando de 2024-07-04 al 2024-07-18
Proceso completado satisfactoriamente en 431.932 segundos
Procesando de 2024-07-19 al 2024-08-02
Proceso completado satisfactoriamente en 123.449 segundos
Procesando de 2024-08-03 al 2024-08-17
Proceso completado satisfactoriamente en 69.006 segundos
Procesando de 2024-08-18 al 2024-09-01
Proceso completado satisfactoriamente en 42.487 segundos
Procesando de 2024-09-02 al 2024-09-16
Proceso completado satisfactoriamente en 32.302 segundos
Procesando de 2024-09-17 al 2024-10-01
Proceso completado satisfactoriamente en 23.369 segundos
Procesando de 2024-10-02 al 2024-10-16
Proceso completado satisfactoriamente en 8.793 segundos
Procesando de 2024-10-17 al 2024-10-31
Proceso completado satisfactoriamente en 6.483 segundos
Procesando de 2024-11-01 al 2024-11-15
Proceso completado satisfactoriamente en 1.802 segundos
Procesando de 2024-11-16 al 2024-11-30
Proceso completado satisfactoriamente en 0.715 segundos
Procesando de 2024-12-01 al 2024-12-15
Pro

In [10]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()